<a href="https://colab.research.google.com/github/nitindavegit/Deep-Learning/blob/main/pytorch07_training_pipeline_using_dataset_and_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

In [24]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


# Feature Engineering

In [25]:
df.shape

(569, 33)

In [26]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)
df.shape

(569, 31)

In [27]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

In [28]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [29]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [30]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

# Creating a custom Dataset

In [31]:
from torch.utils.data import Dataset, DataLoader

class CustomDataSet(Dataset):


  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return len(self.features)

  def __getitem__(self, index):

    return self.features[index], self.labels[index]

# Different dataset for training and validation

In [32]:
train_dataset = CustomDataSet(X_train_tensor, y_train_tensor)
test_dataset = CustomDataSet(X_test_tensor, y_test_tensor)

In [33]:
train_dataset[10]

(tensor([-1.2401, -0.8630, -1.1544, -1.0071,  0.7799,  1.0421,  4.0596,  0.7654,
          2.6905,  4.2541,  1.6587,  2.6586,  0.6668,  0.2556,  1.4102,  3.7881,
         11.4405,  6.7041,  1.8756,  9.2940, -1.0719, -0.9575, -1.0685, -0.8682,
         -0.1350,  0.1374,  2.6565,  0.6655,  0.3458,  2.4305]),
 tensor(0.))

# Different dataloader for training dataset and validating dataset

In [34]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# Neural Network creation using nn Module

In [35]:
class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()

    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [36]:
learning_rate = 0.1
epochs = 25

In [37]:
model = MySimpleNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

loss_function = nn.BCELoss()


###  For each epoch, the train_loader reshuffles the entire dataset and serves it in batches, ensuring the model doesn't memorize the order of samples. For each batch, the model makes predictions, calculates the loss, and backpropagates the error to compute gradients. Finally, the optimizer updates the model weights using those gradients, and this entire process repeats every epoch until the model learns the pattern.

In [38]:
for epoch in range(epochs):
  for batch_features, batch_labels in train_loader:

    y_pred = model(batch_features)

    loss = loss_function(y_pred, batch_labels.view(-1,1))

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()


  print(f"Epoch : {epoch + 1}, loss : {loss.item()}")

Epoch : 1, loss : 0.26679906249046326
Epoch : 2, loss : 0.11330544948577881
Epoch : 3, loss : 0.0658385306596756
Epoch : 4, loss : 0.16028186678886414
Epoch : 5, loss : 0.013561094179749489
Epoch : 6, loss : 0.12325622886419296
Epoch : 7, loss : 0.17683014273643494
Epoch : 8, loss : 0.03611114248633385
Epoch : 9, loss : 0.07513298094272614
Epoch : 10, loss : 0.08461859077215195
Epoch : 11, loss : 0.1298898309469223
Epoch : 12, loss : 0.01461309939622879
Epoch : 13, loss : 0.18261824548244476
Epoch : 14, loss : 0.07243277132511139
Epoch : 15, loss : 0.037262123078107834
Epoch : 16, loss : 0.000986335682682693
Epoch : 17, loss : 0.0541597343981266
Epoch : 18, loss : 0.014225974678993225
Epoch : 19, loss : 0.3356602191925049
Epoch : 20, loss : 0.015801966190338135
Epoch : 21, loss : 0.006657665129750967
Epoch : 22, loss : 0.0032727711368352175
Epoch : 23, loss : 0.5629539489746094
Epoch : 24, loss : 0.12244634330272675
Epoch : 25, loss : 0.01441971305757761


## Model evaluation using test loader

In [39]:
model.eval()   # set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    y_pred = model(batch_features)

    y_pred = (y_pred > 0.8).float()

    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()  # view(-1) = "flatten everything into one single row"

    accuracy_list.append(batch_accuracy)    # making a list of accuracies per batch


# calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)   # taking average of accuracy list

print(f"Accuracy : {overall_accuracy:.4f}")


Accuracy : 0.9427
